# PIDA-RL Diabetes — 02. Entendimiento y calidad de datos

Este notebook inspecciona las fuentes ENSANUT cargadas en el notebook 01 para identificar variables candidatas y evaluar su calidad. **No crea la cohorte, no imputa valores y no modifica archivos fuente.**

El objetivo es producir evidencia auditable para decidir las variables que se utilizarán en `03_preparacion_cohorte.ipynb`.

## 1. Inicialización y recuperación de fuentes

Si ejecutas este notebook justo después del 01, reutilizará `frames` y `metas` presentes en memoria. Si se abrió una sesión nueva, recargará los archivos `.sav` localizados en la carpeta persistente `data/raw`.

In [1]:
from pathlib import Path
import os
import sys
import json
import subprocess
from datetime import datetime, timezone

PROJECT_NAME = 'pida-rl-datos-publicos'
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
PROJECT_ROOT = Path('/content') / PROJECT_NAME

DRIVE_ROOT = next(
    (path for path in [Path('/content/drive'), Path('/content/gdrive')]
     if (path / 'MyDrive').is_dir()),
    None
)

if DRIVE_ROOT is None:
    from google.colab import drive
    mount_point = Path('/content/drive')
    if mount_point.exists() and any(mount_point.iterdir()):
        mount_point = Path('/content/gdrive')
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError(
            f'No es posible montar Drive en {mount_point}; contiene archivos locales. '
            'Reinicia la sesión de Colab o utiliza un punto de montaje vacío.'
        )
    drive.mount(str(mount_point), force_remount=False)
    DRIVE_ROOT = mount_point

MYDRIVE_ROOT = DRIVE_ROOT / 'MyDrive'
PERSISTENT_ROOT = MYDRIVE_ROOT / 'PIDA-RL-Diabetes'
DATA_ROOT = PERSISTENT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
METADATA_DIR = DATA_ROOT / 'metadata'
SRC_ROOT = PROJECT_ROOT / 'src'

for directory in [RAW_DATA_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'La ruta no es un repositorio Git: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('Proyecto:', PROJECT_ROOT)
print('Datos crudos:', RAW_DATA_DIR)
print('Metadatos:', METADATA_DIR)


Mounted at /content/drive
Proyecto: /content/pida-rl-datos-publicos
Datos crudos: /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw
Metadatos: /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata


In [2]:
import numpy as np
import pandas as pd

try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyreadstat'])
    import pyreadstat

SOURCE_SPECS = {
    'adultos': {
        'expected_names': ['AdultosENSANUT2022.sav', 'Adultos_ENSANUT_2022.sav'],
        'keywords': ['adult', 'ensanut', '2022'],
        'required': True
    },
    'antro_tension': {
        'expected_names': ['AntroTensionENSANUT2022.sav', 'Antro_Tension_ENSANUT_2022.sav'],
        'keywords': ['antro', 'tension'],
        'required': False
    },
    'sangre': {
        'expected_names': ['MuestrasSangreENSANUT2022.sav', 'Muestras_Sangre_ENSANUT_2022.sav'],
        'keywords': ['sangre'],
        'required': False
    }
}

def locate_source(spec, search_dir):
    all_files = [path for path in search_dir.rglob('*.sav') if path.is_file()]
    for expected_name in spec['expected_names']:
        exact = [path for path in all_files if path.name.lower() == expected_name.lower()]
        if exact:
            return exact[0]
    keywords = [item.lower() for item in spec['keywords']]
    matches = [
        path for path in all_files
        if all(keyword in path.name.lower() for keyword in keywords)
    ]
    return matches[0] if len(matches) == 1 else None

if 'frames' not in globals() or 'metas' not in globals() or 'adultos' not in frames:
    print('No se encontraron fuentes en memoria; se recargarán desde data/raw.')
    frames = {}
    metas = {}
    for name, spec in SOURCE_SPECS.items():
        path = locate_source(spec, RAW_DATA_DIR)
        if path is None:
            if spec['required']:
                raise FileNotFoundError(
                    f'No se encontró la fuente obligatoria {name!r} en {RAW_DATA_DIR}. '
                    'Ejecuta 01_fuentes_y_carga_reinicio.ipynb o revisa la ubicación del archivo.'
                )
            print(f'{name}: fuente opcional no localizada.')
            continue
        frames[name], metas[name] = pyreadstat.read_sav(path, apply_value_formats=False)
        print(f'{name}: {frames[name].shape}')
else:
    print('Se reutilizan frames y metas de la sesión actual.')

df_adultos = frames['adultos']
df_diabetes = df_adultos

print('Adultos:', df_adultos.shape)
print('Fuentes disponibles:', list(frames.keys()))


No se encontraron fuentes en memoria; se recargarán desde data/raw.
adultos: (11913, 572)
antro_tension: (15073, 99)
sangre: (7737, 46)
Adultos: (11913, 572)
Fuentes disponibles: ['adultos', 'antro_tension', 'sangre']


## 2. Inventario estructural de fuentes

Este inventario identifica dimensiones, nombres de variables, tipos y claves de unión sin interpretar todavía los códigos de las variables.

In [3]:
source_summary = []

for source_name, df in frames.items():
    candidate_keys = [
        column for column in df.columns
        if column.upper() in {'FOLIO_I', 'FOLIO_INT'}
    ]
    source_summary.append({
        'fuente': source_name,
        'filas': int(df.shape[0]),
        'columnas': int(df.shape[1]),
        'claves_detectadas': ', '.join(candidate_keys),
        'duplicados_folio_int': (
            int(df['FOLIO_INT'].duplicated().sum())
            if 'FOLIO_INT' in df.columns else None
        )
    })

df_source_summary = pd.DataFrame(source_summary)
display(df_source_summary)

print('Primeras 60 variables de Adultos:')
print(df_adultos.columns.tolist()[:60])


,fuente,filas,columnas,claves_detectadas,duplicados_folio_int
0,adultos,11913,572,"FOLIO_I, FOLIO_INT",0
1,antro_tension,15073,99,"FOLIO_I, FOLIO_INT",0
2,sangre,7737,46,"FOLIO_I, FOLIO_INT",0


Primeras 60 variables de Adultos:
['FOLIO_I', 'FOLIO_INT', 't_hora', 't_min', 't_sumai', 't_sumaf', 'hora_ini_1', 'fecha_ini_1', 'hora_fin_1', 'fecha_fin_1', 'tiempo1', 'resultado_1', 'hora_ini_2', 'fecha_ini_2', 'hora_fin_2', 'fecha_fin_2', 'tiempo2', 'resultado_2', 'hora_ini_3', 'fecha_ini_3', 'hora_fin_3', 'fecha_fin_3', 'tiempo3', 'resultado_3', 'hora_ini_4', 'fecha_ini_4', 'hora_fin_4', 'fecha_fin_4', 'tiempo4', 'resultado_4', 'hora_ini', 'fecha_ini', 'entidad', 'desc_ent', 'municipio', 'desc_mun', 'nota1', 'edad', 'sexo', 'fech_nac', 'meses', 'asexo', 'aedad', 'nota014', 'a0104', 'a0107', 'a0108', 'a0109', 'A0110A', 'A0110B', 'A0110C', 'A0110D', 'A0110E', 'A0110F', 'A0110G', 'A0110H', 'A0110I', 'a01esp', 'nota015', 'nota016']


## 3. Búsqueda auditable de variables candidatas

Los patrones son solo un mecanismo de descubrimiento. Una variable se incorporará a la cohorte únicamente después de revisar su etiqueta, codificación y significado en el cuestionario/diccionario oficial.

In [4]:
SEARCH_PATTERNS = {
    'identificador': ['folio'],
    'edad': ['edad', 'ed_'],
    'sexo': ['sexo', 'sex_'],
    'diabetes': ['diab', 'diabet', 'gluco', 'insulin'],
    'hipertension': ['hiper', 'tension', 'presion'],
    'complicaciones': ['complic', 'renal', 'retino', 'neurop', 'amputa'],
    'comorbilidades': ['cardio', 'infarto', 'cerebro', 'obes', 'cancer', 'depre'],
    'hospitalizacion': ['hospital', 'urgenc', 'intern'],
    'atencion_salud': ['consulta', 'medic', 'trat', 'control', 'servicio'],
    'antropometria': ['peso', 'talla', 'imc', 'cintura'],
    'laboratorio': ['gluco', 'hba1c', 'hemoglob', 'colesterol', 'triglic']
}

def metadata_label(meta, variable):
    labels = getattr(meta, 'column_labels', None)
    names = getattr(meta, 'column_names', None)
    if labels is not None and names is not None and variable in names:
        return labels[names.index(variable)]
    return None

candidate_rows = []
for source_name, df in frames.items():
    meta = metas.get(source_name)
    for variable in df.columns:
        variable_text = str(variable).lower()
        label = metadata_label(meta, variable) if meta is not None else None
        label_text = str(label).lower() if label else ''
        matched_groups = [
            group for group, patterns in SEARCH_PATTERNS.items()
            if any(pattern in variable_text or pattern in label_text for pattern in patterns)
        ]
        if matched_groups:
            series = df[variable]
            candidate_rows.append({
                'fuente': source_name,
                'variable': variable,
                'etiqueta': label,
                'grupos_candidatos': '; '.join(matched_groups),
                'tipo': str(series.dtype),
                'n_registros': int(series.shape[0]),
                'n_faltantes': int(series.isna().sum()),
                'porcentaje_faltantes': round(float(series.isna().mean() * 100), 2),
                'n_unicos': int(series.nunique(dropna=True))
            })

df_candidates = pd.DataFrame(candidate_rows)
df_candidates = df_candidates.sort_values(['fuente', 'grupos_candidatos', 'variable']).reset_index(drop=True)

print('Variables candidatas encontradas:', len(df_candidates))
display(df_candidates.head(100))


Variables candidatas encontradas: 224


,fuente,variable,etiqueta,grupos_candidatos,tipo,n_registros,n_faltantes,porcentaje_faltantes,n_unicos
0,adultos,A0110A,¿Las acciones que tomó para perder peso fueron...,antropometria,float64,11913,11490,96.45,8
1,adultos,A0110B,¿Las acciones que tomó para perder peso fueron...,antropometria,float64,11913,11652,97.81,8
2,adultos,A0110C,¿Las acciones que tomó para perder peso fueron...,antropometria,float64,11913,11785,98.93,7
3,adultos,A0110D,¿Las acciones que tomó para perder peso fueron...,antropometria,float64,11913,11846,99.44,5
4,adultos,A0110E,¿Las acciones que tomó para perder peso fueron...,antropometria,float64,11913,11869,99.63,4
...,...,...,...,...,...,...,...,...,...
95,adultos,a0306g,"En esa última ocasión, en el lugar en donde lo...",diabetes; atencion_salud,float64,11913,10731,90.08,2
96,adultos,a0307,¿Actualmente toma pastillas o le aplican insul...,diabetes; atencion_salud,float64,11913,10351,86.89,4
97,adultos,a0309a,¿Hace cuánto tiempo inició el tratamiento con ...,diabetes; atencion_salud,float64,11913,11526,96.75,30
98,adultos,a0309m,¿Hace cuánto tiempo inició el tratamiento con ...,diabetes; atencion_salud,float64,11913,11518,96.68,13


## 4. Perfil de calidad de variables candidatas

Se inspeccionan valores frecuentes, intervalos numéricos y posibles códigos especiales. Los valores extremos o códigos como 9, 99, 999, etc. se reportan, pero **no se recodifican en este notebook**.

In [5]:
def profile_variable(df, variable, top_n=10):
    series = df[variable]
    profile = {
        'n': int(len(series)),
        'faltantes': int(series.isna().sum()),
        'porcentaje_faltantes': round(float(series.isna().mean() * 100), 2),
        'n_unicos': int(series.nunique(dropna=True)),
        'tipo': str(series.dtype)
    }

    if pd.api.types.is_numeric_dtype(series):
        finite = series.replace([np.inf, -np.inf], np.nan).dropna()
        profile.update({
            'n_inf': int(np.isinf(series.dropna()).sum()),
            'minimo': float(finite.min()) if not finite.empty else None,
            'p01': float(finite.quantile(0.01)) if not finite.empty else None,
            'mediana': float(finite.median()) if not finite.empty else None,
            'p99': float(finite.quantile(0.99)) if not finite.empty else None,
            'maximo': float(finite.max()) if not finite.empty else None
        })

    frequent = (
        series.value_counts(dropna=False)
        .head(top_n)
        .rename_axis('valor')
        .reset_index(name='frecuencia')
    )
    return profile, frequent

variables_prioritarias = (
    df_candidates[
        df_candidates['fuente'].eq('adultos')
        & df_candidates['grupos_candidatos'].str.contains(
            'edad|sexo|diabetes|complicaciones|comorbilidades|hospitalizacion|atencion_salud',
            regex=True
        )
    ]['variable']
    .drop_duplicates()
    .tolist()
)

print('Variables prioritarias a perfilar:', len(variables_prioritarias))

quality_rows = []
frequent_values = {}

for variable in variables_prioritarias:
    profile, frequent = profile_variable(df_adultos, variable)
    quality_rows.append({'fuente': 'adultos', 'variable': variable, **profile})
    frequent_values[variable] = frequent.to_dict(orient='records')

df_quality = pd.DataFrame(quality_rows)
display(df_quality.sort_values(['porcentaje_faltantes', 'variable']).head(100))


Variables prioritarias a perfilar: 154


,fuente,variable,n,faltantes,porcentaje_faltantes,n_unicos,tipo,n_inf,minimo,p01,mediana,p99,maximo
49,adultos,A0605A,11913,0,0.00,6,object,NaN,NaN,NaN,NaN,NaN,NaN
50,adultos,A0605B,11913,0,0.00,4,object,NaN,NaN,NaN,NaN,NaN,NaN
51,adultos,A0605C,11913,0,0.00,3,object,NaN,NaN,NaN,NaN,NaN,NaN
52,adultos,A0605D,11913,0,0.00,2,object,NaN,NaN,NaN,NaN,NaN,NaN
53,adultos,A0607A1,11913,0,0.00,6,object,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,adultos,a1002h,11913,10936,91.80,9,float64,0.0,1.0,1.0,4.0,9.0,9.0
135,adultos,a1302,11913,10942,91.85,47,float64,0.0,12.0,12.0,18.0,98.0,98.0
23,adultos,a1002d,11913,10964,92.03,9,float64,0.0,1.0,1.0,3.0,9.0,9.0
57,adultos,a0607a,11913,11000,92.34,62,float64,0.0,0.0,0.0,0.0,2000.0,3900.0


## 5. Revisar etiquetas y valores frecuentes

Usa esta celda para revisar las variables candidatas antes de seleccionar la especificación de cohorte. Cambia `VARIABLES_A_REVISAR` por nombres reales tomados de la tabla previa.

In [6]:
VARIABLES_A_REVISAR = variables_prioritarias[:20]

for variable in VARIABLES_A_REVISAR:
    label = metadata_label(metas.get('adultos'), variable)
    profile, frequent = profile_variable(df_adultos, variable, top_n=15)
    print('\n' + '=' * 90)
    print('Variable:', variable)
    print('Etiqueta:', label)
    print('Perfil:', profile)
    display(frequent)



Variable: A0312A
Etiqueta: Actualmente, ¿qué otro tratamiento lleva para controlar su azúcar?
Perfil: {'n': 11913, 'faltantes': 10351, 'porcentaje_faltantes': 86.89, 'n_unicos': 5, 'tipo': 'float64', 'n_inf': 0, 'minimo': 1.0, 'p01': 1.0, 'mediana': 4.0, 'p99': 5.0, 'maximo': 5.0}


,valor,frecuencia
0,NaN,10351
1,5.0,779
2,1.0,553
3,3.0,94
4,2.0,94
5,4.0,42



Variable: A0312B
Etiqueta: Actualmente, ¿qué otro tratamiento lleva para controlar su azúcar?
Perfil: {'n': 11913, 'faltantes': 11595, 'porcentaje_faltantes': 97.33, 'n_unicos': 3, 'tipo': 'float64', 'n_inf': 0, 'minimo': 2.0, 'p01': 2.0, 'mediana': 2.0, 'p99': 4.0, 'maximo': 4.0}


,valor,frecuencia
0,NaN,11595
1,2.0,164
2,3.0,141
3,4.0,13



Variable: A0312C
Etiqueta: Actualmente, ¿qué otro tratamiento lleva para controlar su azúcar?
Perfil: {'n': 11913, 'faltantes': 11810, 'porcentaje_faltantes': 99.14, 'n_unicos': 2, 'tipo': 'float64', 'n_inf': 0, 'minimo': 3.0, 'p01': 3.0, 'mediana': 3.0, 'p99': 4.0, 'maximo': 4.0}


...

,valor,frecuencia
0,NaN,10411
1,1.0,614
2,4.0,284
3,8.0,267
4,6.0,166
5,3.0,103
6,9.0,27
7,5.0,13
8,7.0,12
9,2.0,12



Variable: a0407esp
Etiqueta: ¿En dónde se atiende principalmente para controlar su presión alta? Especifique
Perfil: {'n': 11913, 'faltantes': 0, 'porcentaje_faltantes': 0.0, 'n_unicos': 28, 'tipo': 'object'}


,valor,frecuencia
0,,11886
1,CARAVANA DE SALUD,1
2,HOSPITAL DE ESTADOS UNIDOS,1
3,CLINICA DEL MAGISTERIO,1
4,DOCTOR EN EL TRABAJO,1
5,DOCTOR NATURISTA,1
6,MÉDICO LABORAL,1
7,INSTITUTO MUNICIPAL DE PENSIONES,1
8,PENSIONES MUNICIPALES,1
9,PENSIONES,1



Variable: a0502b
Etiqueta: ¿Le ha dicho el médico que usted tiene (o tuvo) angina de pecho (dolor o molestia en el pecho, que desaparece regularmente de forma espontánea con el reposo o con medicinas?
Perfil: {'n': 11913, 'faltantes': 0, 'porcentaje_faltantes': 0.0, 'n_unicos': 2, 'tipo': 'float64', 'n_inf': 0, 'minimo': 1.0, 'p01': 1.0, 'mediana': 2.0, 'p99': 2.0, 'maximo': 2.0}


,valor,frecuencia
0,2.0,11695
1,1.0,218


## 6. Exportar evidencia para la preparación de cohorte

Los archivos exportados no contienen microdatos. Sirven para documentar la selección posterior de variables y las decisiones de recodificación.

In [7]:
candidates_path = METADATA_DIR / 'diccionario_variables_candidatas.csv'
quality_path = METADATA_DIR / 'perfil_calidad_variables_candidatas.csv'
frequent_path = METADATA_DIR / 'valores_frecuentes_variables_candidatas.json'
sources_path = METADATA_DIR / 'resumen_fuentes.csv'

df_candidates.to_csv(candidates_path, index=False, encoding='utf-8-sig')
df_quality.to_csv(quality_path, index=False, encoding='utf-8-sig')
df_source_summary.to_csv(sources_path, index=False, encoding='utf-8-sig')

with frequent_path.open('w', encoding='utf-8') as file:
    json.dump(frequent_values, file, ensure_ascii=False, indent=2, default=str)

print('Archivos generados:')
for path in [candidates_path, quality_path, frequent_path, sources_path]:
    print(' -', path, '| existe:', path.is_file())


Archivos generados:
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/diccionario_variables_candidatas.csv | existe: True
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/perfil_calidad_variables_candidatas.csv | existe: True
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/valores_frecuentes_variables_candidatas.json | existe: True
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/metadata/resumen_fuentes.csv | existe: True


## 7. Decisión requerida antes del notebook 03

Antes de construir la cohorte, documenta una especificación explícita con los nombres reales de las variables seleccionadas, la codificación de respuestas válidas y el tratamiento de faltantes/códigos especiales.

En la siguiente etapa no se deben usar variables solo por coincidencia de nombre. La selección deberá basarse en el diccionario oficial de ENSANUT y en el perfil de calidad exportado.